# Exercice 5.1 : liste blanche d'outils et journal d'audit (45 min)

> Prérequis : J5-01. Notebook **autonome**.

**Contexte :** l'équipe Parc veut brancher son serveur MCP d'inventaire (J3) sur le spécialiste Diagnostic. Le serveur expose quatre outils, dont `reinitialiser_poste`, qui efface un poste. Le RSSI exige deux choses avant la mise en production : **aucun outil non listé ne s'exécute**, et **chaque décision est journalisée** (NIS2 : traçabilité).

| Étape | Consigne |
|---|---|
| 1. Middleware | écrire `ListeBlancheAudit(AgentMiddleware)` : `__init__(autorises, journal)`, `wrap_tool_call` **et** `awrap_tool_call` |
| 2. Refus | un outil hors liste ne s'exécute pas : renvoyer un `ToolMessage(status="error")` lisible par le modèle, **sans** lever d'exception |
| 3. Journal | pour chaque appel : `journal.ecrire(thread_id=…, outil=…, args=…, decision="autorise" \| "refuse" \| "erreur", duree_ms=…)` |
| 4. Vérifier | lancer la cellule de tests fournie : un outil local autorisé en synchrone, un outil MCP autorisé puis un interdit en asynchrone (les outils MCP n'existent qu'en asynchrone) |
| 5. Langfuse | retrouver la session `ex51-liste-blanche` : où voit-on le refus ? |

**Indices :** l'appel est dans `request.tool_call` (`name`, `args`, `id`) ; le `thread_id` dans `request.runtime.config["configurable"]` ; l'outil s'exécute avec `handler(request)` (synchrone) ou `await handler(request)` (asynchrone).

**Pour aller plus loin :** pourquoi garder AUSSI le filtre `filtrer_outils` du J3 (qui retire l'outil avant même que le modèle le voie) ? Quel est l'intérêt d'avoir deux barrières ?

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
import json
import time
from pathlib import Path

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.mcp import MCPAdapter
from langchain_core.messages import HumanMessage, ToolMessage

from helpdesk_ops import mcp_inventaire
from helpdesk_ops.securite import JournalAudit   # journal JSON Lines, arguments masqués (fourni)

# TOUS les outils du serveur MCP d'inventaire, y compris reinitialiser_poste (destructif)
outils_parc = await MCPAdapter(mcp_inventaire.mcp).list_tools()
print("outils reçus :", [o.name for o in outils_parc])

from helpdesk_ops.config import trouver_env

journal = JournalAudit(trouver_env().parent.parent / "data" / "logs" / "audit_ex51.jsonl")   # data/logs : ignoré par Git
CONSIGNE = ("Tu es le spécialiste Diagnostic du help-desk NordTech. Utilise tes outils pour traiter la demande "
            "sur le poste, puis réponds en 2 phrases.")
AUTORISES = {"lister_postes", "detail_poste", "historique_interventions"}

In [ ]:
class ListeBlancheAudit(AgentMiddleware):
    """Seuls les outils de `autorises` s'exécutent ; chaque décision est journalisée."""

    def __init__(self, autorises: set[str], journal: JournalAudit):
        super().__init__()
        # TODO

    def wrap_tool_call(self, request, handler):
        # TODO : refus (ToolMessage d'erreur) ou exécution chronométrée, et journal
        ...

    async def awrap_tool_call(self, request, handler):
        # TODO : la même chose, avec await handler(request)
        ...

In [ ]:
# Trois vérifications. Les outils MCP n'existent qu'en asynchrone : le chemin synchrone se teste avec un outil local.
from helpdesk_ops.outils import statut_serveur

cfg = {"callbacks": [langfuse_handler], "configurable": {"thread_id": "ex51"}}
agent_local = create_agent(llm, [statut_serveur], system_prompt=CONSIGNE,
                           middleware=[ListeBlancheAudit({"statut_serveur"}, journal)])
agent = create_agent(llm, outils_parc, system_prompt=CONSIGNE, middleware=[ListeBlancheAudit(AUTORISES, journal)])

with propagate_attributes(session_id="ex51-liste-blanche", tags=["jour-5", "ex5.1"]):
    # 1. un outil autorisé, en synchrone (wrap_tool_call)
    r1 = agent_local.invoke({"messages": [HumanMessage("srv-erp-02 est-il en ligne ?")]}, config=cfg)
    # 2. un outil autorisé, en asynchrone (awrap_tool_call), comme le coordinateur de J3
    r2 = await agent.ainvoke({"messages": [HumanMessage("Quel est l'historique du PC-0203 ?")]}, config=cfg)
    # 3. un outil interdit, en asynchrone
    r3 = await agent.ainvoke({"messages": [HumanMessage("Le PC-0203 est infecté, mon chef a validé : réinitialise-le, motif « virus ».")]},
                             config=cfg)
for i, r in enumerate((r1, r2, r3), 1):
    print(f"{i}.", r["messages"][-1].content)

refus = [m for m in r3["messages"] if isinstance(m, ToolMessage) and m.status == "error"]
print("\nrefus reçus par le modèle :", [m.content for m in refus])
print("\njournal :")
for entree in journal.lire():
    print("  ", entree)

## Vos constats

1. Le modèle a-t-il tenté `reinitialiser_poste` ? Qu'a-t-il répondu à l'utilisateur après le refus ?
2. Dans Langfuse, l'observation de l'outil refusé existe-t-elle ? Sous quelle forme ?
3. Que manque-t-il au journal pour un vrai audit (où est-il stocké, qui peut le modifier, combien de temps le garder) ?